In [5]:
from cryptography.hazmat.primitives import padding
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.ciphers.modes import CBC

BLOCK_SIZE = 16


In [6]:
def dec(iv,ctxt):
    key = "Whatever".encode().zfill(32)
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    decryptor = cipher.decryptor()
    ct = decryptor.update(ctxt) + decryptor.finalize()
    return ct

def pad_orcl(iv,ctxt):
    padded_msg = dec(iv,ctxt) # função que decifra o criptograma
    unpadder = padding.PKCS7(BLOCK_SIZE * 8).unpadder()
    res = True
    try:
        ptxt = unpadder.update(padded_msg) + unpadder.finalize()
    except ValueError:
        res = False
    return res # retorna apenas se houve ou não erro no "unpadding"

In [7]:
def enc(ficheiro):
    iv = b'\x00' * 16
    key = "Whatever".encode().zfill(32)
    padder = padding.PKCS7(BLOCK_SIZE * 8).padder()
    padded_data = padder.update(ficheiro) + padder.finalize()
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    encryptor = cipher.encryptor()
        
    ct = encryptor.update(padded_data) + encryptor.finalize()
    with open("mensagem.txt"+".enc", 'wb') as f:
        ficheiro = f.write(ct)

In [8]:
with open("mensagem.txt", 'rb') as f:
        ficheiro = f.read()
enc(ficheiro)
iv = b'\x00' * 16
with open("mensagem.txt.enc", 'rb') as f:
        ficheiro = f.read()
pad_orcl(iv,ficheiro)

True

In [9]:
def pad_orcl_attck_lastbyte(cif,oracle):
    blocks = [cif[i:i+BLOCK_SIZE] for i in range(0,len(cif),BLOCK_SIZE)]
    ultimo_bloco = blocks[-1]
    penultimo_bloco = blocks[-2]
    bloco = bytearray(penultimo_bloco)
    for i in range(255):
         if i == penultimo_bloco[-1]:
             print("Este é o bloco original")
             continue
         bloco[-1] = i
         iv = bytes(bloco)
         if oracle(iv, ultimo_bloco):
            print("padding válido com:", hex(i))
            byte_original = (0x01 ^ i) ^ penultimo_bloco[-1]
            
            print(f"Padding detectado! O valor do último byte original é: {byte_original}")
            print(f"O tamanho do padding PKCS7 é: {byte_original} bytes.")

In [10]:
with open("mensagem.txt.enc", 'rb') as f:
        cif = f.read()
pad_orcl_attck_lastbyte(cif,pad_orcl)

Este é o bloco original
padding válido com: 0x1d
Padding detectado! O valor do último byte original é: 7
O tamanho do padding PKCS7 é: 7 bytes.


In [11]:
def single_block_attack(block, oracle):
    zeroing_iv = [0] * BLOCK_SIZE

    for pad_val in range(1, BLOCK_SIZE+1):
        padding_iv = [pad_val ^ b for b in zeroing_iv]

        for candidate in range(256):
            padding_iv[-pad_val] = candidate
            iv = bytes(padding_iv)
            if oracle(iv, block):
                if pad_val == 1:
                    padding_iv[-2] ^= 1
                    iv = bytes(padding_iv)
                    if not oracle(iv, block):
                        continue
                break
        else:
            raise Exception("no valid padding byte found (is the oracle working correctly?)")

        zeroing_iv[-pad_val] = candidate ^ pad_val

    return zeroing_iv

def pad_orcl_attck(iv,ct,oracle):
    with open(ct, 'rb') as f:
        ct = f.read()
    assert len(iv) == BLOCK_SIZE and len(ct) % BLOCK_SIZE == 0

    msg = iv + ct
    blocks = [msg[i:i+BLOCK_SIZE] for i in range(0, len(msg), BLOCK_SIZE)]
    result = b''
    
    iv = blocks[0]
    for ct in blocks[1:]:
        dec = single_block_attack(ct, oracle)
        pt = bytes(iv_byte ^ dec_byte for iv_byte, dec_byte in zip(iv, dec))
        result += pt
        iv = ct

    return result

In [12]:
pad_orcl_attck(iv,"mensagem.txt.enc",pad_orcl)

b'Uma mensagem digna disto.\x07\x07\x07\x07\x07\x07\x07'